<a href="https://colab.research.google.com/github/osmarbraz/sri/blob/main/1_1_Segmentacao_Limpeza_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segmentação e limpeza

Realiza a segmentação e limpeza de um conjunto de dados em sua forma bruta.

**Entrada**: "`documentos.csv`".
 - Cada linha do arquivo é formado por `["id","documento"]`.
  - `"id"` é o idenficador do documento.
  - `"documento"` o documento texto no seu formato bruto.

**Saída**: "`dataset.zip`".
 - Dentro do arquivo compactado `dataset.zip` está o arquivo `dataset.csv`. Cada linha de `dataset.csv` é formado por `["id","sentencas","documento"]`.
  - `"id"` é o idenficador do documento na base de dados.
  - `"sentencas"` é uma lista com as sentenças do documento.
  - `"documento"` o documento limpo, mas não segmentado.

**Processamento**:
1. Copia o arquivo "`documentos.csv`" para a máquina local do
Google Colab.
2. Realiza as seguintes tarefas de limpeza:
    - Normalização de codificação e decodificação de html entities (ftfy);
    - Substitui \n por espaço em branco;
    - Elimina pontuações repetidas ("???","!!!",",,,");
    - Elimina de espaços em branco repetidos;
    - Elimina tags;
    - **Local para outras operações de limpeza.**
3. Descarta documentos que geram mais de 512 tokens pelo tokenizador do BERT.
4. Realiza a segmentação.
5. Gera o arquivo "`dataset.csv`" com os dados pré-processados.
6. Compacta o arquivo "`dataset.csv`" para "`dataset.zip`".
7. Copia o arquivo "`dataset.zip`" para o google drive.

**Atenção - Para executar este notebook e os próximos siga as instruções abaixo.**

1. Crie a pasta "`Colab Notebooks`" na raiz do seu google drive para receber pastas de projetos.
2. Dentro da pasta "`Colab Notebooks`" crie a pasta "`SRI`" para armazenar e executar os notebooks das atividades práticas da disciplina.
3. Dentro da pasta "`SRI`" crie a pasta "data" e coloque o arquivo de dados "`documentos.csv`".

**Testes:**

Os arquivos "`documentos.csv`" e "`dataset.csv`" precisam ter ao final da execução a mesma quantidade de linhas. Onde cada linha de  "`dataset.csv`" esta relacionada com a mesma linha de "`documentos.csv`".


# 1 Preparação do ambiente

Preparação do ambiente para execução do script.

## 1.1 Tempo inicial de processamento

In [1]:
# Import das bibliotecas.
import time
import datetime

# Marca o tempo de início do processamento
inicio_processamento = time.time()

## 1.2 Funções e classes auxiliares

Verifica se existe o diretório do notebook no diretório corrente.   


In [2]:
# Import das bibliotecas.
import os # Biblioteca para manipular arquivos

# ============================
def verificaDiretorioNotebook():
    """
      Verifica se existe o diretório do notebook no diretório corrente.
    """

    # Verifica se o diretório existe
    if not os.path.exists(DIRETORIO_NOTEBOOK):
        # Cria o diretório
        os.makedirs(DIRETORIO_NOTEBOOK)
        logging.info("Diretório do notebook criado: {}".format(DIRETORIO_NOTEBOOK))

    return DIRETORIO_NOTEBOOK

Remove tags de um documento

In [3]:
def remove_tags(documento):
    """
      Remove tags de um documento
    """

    import re

    documento_limpo = re.compile("<.*?>")
    return re.sub(documento_limpo, "", documento)

Função auxiliar para formatar o tempo como `hh: mm: ss`

In [4]:
# Import das bibliotecas.
import time
import datetime

def formataTempo(tempo):
    """
      Pega a tempo em segundos e retorna uma string hh:mm:ss
    """
    # Arredonda para o segundo mais próximo.
    tempoArredondado = int(round((tempo)))

    # Formata como hh:mm:ss
    return str(datetime.timedelta(seconds=tempoArredondado))

Classe(ModelosParametros) de definição dos parâmetros dos modelos

In [5]:
# Import das bibliotecas.
from dataclasses import dataclass, field
from typing import Dict, Optional
from typing import List

@dataclass
class ModelosParametros:
    modelo_spacy: str = field(
        default="pt_core_news_lg",
        metadata={"help": "nome do modelo do spaCy."},
    )

    sentenciar_documento: bool = field(
        default=True,
        metadata={"help": "Dividir o documento em sentenças(frases)."},
    )

    max_seq_len: Optional[int] = field(
      default=None,
      metadata={'help': 'max seq len'},
    )

    pretrained_model_name_or_path: str = field(
      default='neuralmind/bert-base-portuguese-cased',
      metadata={'help': 'nome do modelo pré-treinado do BERT.'},
    )

    do_lower_case: bool = field(
      default=False,
      metadata={'help': 'define se o texto do modelo deve ser todo em minúsculo.'},
    )
    output_attentions: bool = field(
      default=False,
      metadata={'help': 'habilita se o modelo retorna os pesos de atenção.'},
    )
    output_hidden_states: bool = field(
      default=False,
      metadata={'help': 'habilita gerar as camadas ocultas do modelo.'},
    )

Biblioteca de limpeza de tela


In [6]:
# Import das bibliotecas.
from IPython.display import clear_output

## 1.3 Tratamento de logs

In [7]:
# Import das bibliotecas.
import logging # Biblioteca de logging

# Formatando a mensagem de logging
logging.basicConfig(format="%(asctime)s : %(levelname)s : %(message)s")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

## 1.4 Identificando o ambiente Colab

In [8]:
# Import das bibliotecas.
import sys # Biblioteca para acessar módulos do sistema

# Se estiver executando no Google Colaboratory
# Retorna true ou false se estiver no Google Colaboratory
IN_COLAB = "google.colab" in sys.modules

## 1.5 Colaboratory

Usando Colab GPU para Treinamento


Uma GPU pode ser adicionada acessando o menu e selecionando:

`Edit -> Notebook Settings -> Hardware accelerator -> (GPU)`

Em seguida, execute a célula a seguir para confirmar que a GPU foi detectada.

In [9]:
# Import das bibliotecas.
import tensorflow as tf

# Recupera o nome do dispositido da GPU.
device_name = tf.test.gpu_device_name()

# O nome do dispositivo deve ser parecido com o seguinte:
if device_name == "/device:GPU:0":
    logging.info("Encontrei GPU em: {}".format(device_name))
else:
    logging.info("Dispositivo GPU não encontrado")
    #raise SystemError("Dispositivo GPU não encontrado")

INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:root:Dispositivo GPU não encontrado


Nome da GPU

Para que a torch use a GPU, precisamos identificar e especificar a GPU como o dispositivo. Posteriormente, em nosso ciclo de treinamento, carregaremos dados no dispositivo.

Vale a pena observar qual GPU você recebeu. A GPU Tesla P100 é muito mais rápido que as outras GPUs, abaixo uma lista ordenada:
- 1o Tesla P100
- 2o Tesla T4
- 3o Tesla P4 (Não tem memória para execução 4 x 8, somente 2 x 4)
- 4o Tesla K80 (Não tem memória para execução 4 x 8, somente 2 x 4)

In [10]:
# Import das bibliotecas.
import torch

def getDeviceGPU():
    """
      Retorna um dispositivo de GPU se disponível ou CPU.

      Retorno:
        `device` - Um device de GPU ou CPU.
    """

    # Se existe GPU disponível.
    if torch.cuda.is_available():

        # Diz ao PyTorch para usar GPU.
        device = torch.device("cuda")

        logging.info("Existem {} GPU(s) disponíveis.".format(torch.cuda.device_count()))
        logging.info("Iremos usar a GPU: {}.".format(torch.cuda.get_device_name(0)))

    # Se não.
    else:
        logging.info("Sem GPU disponível, usando CPU.")
        device = torch.device("cpu")

    return device

In [11]:
# Recupera o device com GPU ou CPU
device = getDeviceGPU()

INFO:root:Sem GPU disponível, usando CPU.


Memória

Memória disponível no ambiente

In [12]:
# Importando as bibliotecas.
from psutil import virtual_memory

ram_gb = virtual_memory().total / 1e9
logging.info("Seu ambiente de execução tem {: .1f} gigabytes de RAM disponível\n".format(ram_gb))

if ram_gb < 20:
  logging.info("Para habilitar um tempo de execução de RAM alta, selecione menu o ambiente de execução> \"Alterar tipo de tempo de execução\"")
  logging.info("e selecione High-RAM. Então, execute novamente está célula")
else:
  logging.info("Você está usando um ambiente de execução de memória RAM alta!")

INFO:root:Seu ambiente de execução tem  13.6 gigabytes de RAM disponível

INFO:root:Para habilitar um tempo de execução de RAM alta, selecione menu o ambiente de execução> "Alterar tipo de tempo de execução"
INFO:root:e selecione High-RAM. Então, execute novamente está célula


## 1.6 Monta uma pasta no google drive para carregar os arquivos de dados.

In [13]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # import necessário
  from google.colab import drive

  # Monta o drive na pasta especificada
  drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.7 Instalação do spaCy

https://spacy.io/

Modelos do spaCy para português:
https://spacy.io/models/pt

Uso:
https://spacy.io/usage

In [14]:
# Instala dependências do spacy
!pip install -U pip==25.3 setuptools==80.9.0 wheel==0.45.1

In [15]:
# Instala uma versão específica
!pip install -U spacy==3.8.11

  Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (27 kB)
Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)


## 1.8 Instalação do BERT

Instala a interface pytorch para o BERT by Hugging Face.

https://huggingface.co/docs/transformers/installation

In [16]:
!pip install -U transformers==4.49.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 78.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.6 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [huggingface-hub]    WARNING: Ignoring invalid distribution ~pacy (/usr/local/lib/python3.12/dist-packages)
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [tokenizers]    WARNING: Ignoring invalid distribution ~pacy (/usr/local/lib/python3.12/dist-packages)
    Found existing installation: transformers 5.0.0
    Uninstalli

## 1.9 Instalação do ftfy

Realiza a normalização do texto devido a problemas de codificação de caracteres e decodificação de html entities.

https://pypi.org/project/ftfy/

In [17]:
!pip install ftfy==6.3.1

# 2 Parametrização

## Gerais

In [18]:
# Definição dos parâmetros gerais

## Específicos

Parâmetros do modelo

In [19]:
# Definição dos parâmetros do Modelo.
model_args = ModelosParametros(

    modelo_spacy = "pt_core_news_lg",
    #modelo_spacy = "pt_core_news_md",
    #modelo_spacy = "pt_core_news_sm",

    sentenciar_documento = True,

    max_seq_len = 512,

    #pretrained_model_name_or_path = "neuralmind/bert-large-portuguese-cased",
    pretrained_model_name_or_path = "neuralmind/bert-base-portuguese-cased",

    do_lower_case = False,  # default True
    output_attentions = False,  # default False
    output_hidden_states = True, # default False
)

## Nome do diretório dos arquivos de dados

In [20]:
# Diretório do notebook
DIRETORIO_NOTEBOOK = "SRI"

## Define o caminho para os arquivos de dados

In [21]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # Diretorio local para os arquivos de dados
  DIRETORIO_LOCAL = "/content/" + DIRETORIO_NOTEBOOK + "/"

  # Diretorios possiveis no Google Drive com os arquivos de dados.
  # O primeiro e o padrao original do notebook; os demais cobrem a pasta SRI na raiz do Drive.
  DIRETORIO_DRIVE_CANDIDATOS = [
      "/content/drive/MyDrive/Colab Notebooks/" + DIRETORIO_NOTEBOOK + "/data/",
      "/content/drive/MyDrive/Colab Notebooks/" + DIRETORIO_NOTEBOOK + "/",
      "/content/drive/MyDrive/" + DIRETORIO_NOTEBOOK + "/data/",
      "/content/drive/MyDrive/" + DIRETORIO_NOTEBOOK + "/",
  ]

  # Usa o primeiro diretorio existente. Se nenhum existir, mantem o padrao original para mensagens claras.
  DIRETORIO_DRIVE = next((d for d in DIRETORIO_DRIVE_CANDIDATOS if os.path.exists(d)), DIRETORIO_DRIVE_CANDIDATOS[0])
else:

  # Diretorio local para os arquivos de dados
  DIRETORIO_LOCAL = "./data/"

  # Diretorio com os arquivos de dados
  DIRETORIO_DRIVE = "./data/"
  DIRETORIO_DRIVE_CANDIDATOS = [DIRETORIO_DRIVE]


In [22]:
# Funcoes para localizar e copiar arquivos entre Google Drive e diretorio local.
import os
import shutil

# ============================
def localizar_arquivo_drive(nome_arquivo):
    """
      Localiza um arquivo nos diretorios configurados do Google Drive.
      Se nao encontrar nos caminhos candidatos, procura em todo o MyDrive.
    """

    caminhos_verificados = []

    for diretorio in DIRETORIO_DRIVE_CANDIDATOS:
        caminho = os.path.join(diretorio, nome_arquivo)
        caminhos_verificados.append(caminho)
        if os.path.exists(caminho):
            return caminho

    raiz_busca = "/content/drive/MyDrive" if IN_COLAB else DIRETORIO_DRIVE
    if os.path.exists(raiz_busca):
        for raiz, _, arquivos in os.walk(raiz_busca):
            if nome_arquivo in arquivos:
                return os.path.join(raiz, nome_arquivo)

    raise FileNotFoundError(
        "Arquivo nao encontrado: {}\nCaminhos verificados:\n{}".format(
            nome_arquivo,
            "\n".join(caminhos_verificados)
        )
    )

# ============================
def copiar_do_drive_para_local(nome_arquivo):
    """
      Copia um arquivo do Google Drive para DIRETORIO_LOCAL.
    """

    origem = localizar_arquivo_drive(nome_arquivo)
    os.makedirs(DIRETORIO_LOCAL, exist_ok=True)
    destino = os.path.join(DIRETORIO_LOCAL, nome_arquivo)
    shutil.copy2(origem, destino)
    logging.info("Arquivo copiado: {} -> {}".format(origem, destino))
    return destino

# ============================
def copiar_do_local_para_drive(nome_arquivo):
    """
      Copia um arquivo de DIRETORIO_LOCAL para DIRETORIO_DRIVE.
    """

    origem = os.path.join(DIRETORIO_LOCAL, nome_arquivo)
    if not os.path.exists(origem):
        raise FileNotFoundError("Arquivo local nao encontrado: {}".format(origem))

    os.makedirs(DIRETORIO_DRIVE, exist_ok=True)
    destino = os.path.join(DIRETORIO_DRIVE, nome_arquivo)
    shutil.copy2(origem, destino)
    logging.info("Arquivo copiado: {} -> {}".format(origem, destino))
    return destino


# 3 spaCy

## 3.1 Download arquivo modelo

Uso:
https://spacy.io/usage

Modelos:
https://spacy.io/models

In [23]:
!python -m spacy download $model_args.modelo_spacy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 14.2 MB/s  0:00:20
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 3.2 Carrega o modelo

In [24]:
# Import das bibliotecas.
import spacy # Biblioteca do spaCy

nlp = spacy.load(model_args.modelo_spacy)

# 4 Carregando BERT

## 4.1 Carregando o modelo Pré-treinado BERT

Lista de modelos da comunidade:
* https://huggingface.co/models

Português(https://github.com/neuralmind-ai/portuguese-bert):
* **"neuralmind/bert-base-portuguese-cased"**
* **"neuralmind/bert-large-portuguese-cased"**

In [25]:
# Import das bibliotecas
from transformers import BertModel

# Carrega o modelo
#model = BertModel.from_pretrained(model_args.pretrained_model_name_or_path,
#                                  output_attentions=model_args.output_attentions,
#                                  output_hidden_states=model_args.output_hidden_states)

## 4.2 Carregando o tokenizador BERT

O tokenizador utiliza WordPiece, veja em [artigo original](https://arxiv.org/pdf/1609.08144.pdf).

Por default(`do_lower_case=True`) todas as letras são colocadas para minúsculas. Para ignorar a conversão para minúsculo use o parâmetro `do_lower_case=False`. Esta opção também considera as letras acentuadas(ãçéí...), que são necessárias a língua portuguesa.

O parâmetro `do_lower_case` interfere na quantidade tokens a ser gerado a partir de um documento. Quando igual a `False` reduz a quantidade de tokens gerados.

In [26]:
# Import das bibliotecas
from transformers import BertTokenizer

# Carrega o tokenizador
tokenizer = BertTokenizer.from_pretrained(model_args.pretrained_model_name_or_path,
                                          do_lower_case=model_args.do_lower_case)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

# 5 Pré-processamento do arquivo "`documentos.csv`"


## 5.1 Carrega os dados bruto

### 5.1.1 Especifica nome do arquivo de dados bruto



In [27]:
# Nome do arquivo
NOME_ARQUIVO_DADOS_BRUTO = "documentos.csv"

### 5.1.2 Cria o diretório local para receber os arquivos de dados

In [28]:
# Importando as bibliotecas.
import os

# Se estiver executando no Google Colaboratory
if IN_COLAB:

  # Cria o diretório para receber os arquivos Originais e Permutados
  # Diretório a ser criado
  dirbase = DIRETORIO_LOCAL[:-1]

  if not os.path.exists(dirbase):
      # Cria o diretório
      os.makedirs(dirbase)
      logging.info("Diretório criado: {}".format(dirbase))
  else:
      logging.info("Diretório já existe: {}".format(dirbase))

INFO:root:Diretório criado: /content/SRI


### 5.1.3 Copia o arquivos de dados bruto do google drive para o diretório local

In [29]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  copiar_do_drive_para_local(NOME_ARQUIVO_DADOS_BRUTO)

  logging.info("Terminei a copia!")


INFO:root:Arquivo copiado: /content/drive/MyDrive/Colab Notebooks/SRI/data/documentos.csv -> /content/SRI/documentos.csv
INFO:root:Terminei a copia!


### 5.1.4 Carrega os dados no formato bruto

Atributos do arquivo **documentos**:
0. "id"
1. "documento"

In [30]:
# Import das bibliotecas.
import pandas as pd

# Abre o arquivo e retorna o DataFrame
df_documentos_bruto = pd.read_csv(DIRETORIO_LOCAL + NOME_ARQUIVO_DADOS_BRUTO, sep=";", encoding="UTF-8")

logging.info("TERMINADO FONTES: {}.".format(len(df_documentos_bruto)))

INFO:root:TERMINADO FONTES: 40.


In [31]:
df_documentos_bruto.sample(5)

,id,documento
1,2,Este é um exemplo do motivo pelo qual a maiori...
14,15,Este filme é absolutamente terrível e horrível...
3,4,Nem mesmo os Beatles puderam escrever músicas ...
9,10,Fazendeiros ricos em Buenos Aires têm uma long...
35,36,Todas as críticas deste filme são bastante vál...


## 5.2 Segmentação e limpeza dos dados bruto

### 5.2.1 Limpeza do documento

- Normalização de codificação e decodificação de html entities (ftfy);
- Substitui \n por espaço em branco;
- Eliminar pontuações repetidas ("???","!!!",",,,");
- Eliminar de espaços em branco repetidos;
- Eliminar tags;
- **>>> Outras operações de limpeza podem ser adicionadas aqui.<<<**.


Limpeza de tags html: https://pypi.org/project/beautifulsoup4/ ou
https://medium.com/pyladiesbh/beautiful-soup-parseamento-de-html-337197a7d4b9


In [32]:
# Import das bibliotecas,
import ftfy # Biblioteca para normalização do texto
import re # Biblioteca para expressões regulares
from tqdm.notebook import tqdm as tqdm_notebook # Biblioteca para barra de progresso

# Lista para os documentos tratados
documentos_tratados = []

# Acumula o total de sentenças
conta_barra_n = 0
conta_virgulas = 0
conta_interrogacoes = 0
conta_exclamacoes = 0
conta_tags = 0
conta_espacos = 0

logging.info("Processando {} documentos.".format(len(df_documentos_bruto)))

# Barra de progresso dos dados
dados_bruto_bar = tqdm_notebook(df_documentos_bruto.iterrows(), desc=f"Dados", unit=f"registro", total=len(df_documentos_bruto))

# Percorre os registros dos dados bruto
for (i, linha) in dados_bruto_bar:

  # Recupera o id documento
  id_documento = linha.values[0]

  # Recupera o documento
  documento = str(linha.values[1])

  # Transforma em string e remove os espaços do início e do fim
  documento = str(documento).strip()

  # Normalização com o ftfy, e conversão de html entities em caracteres acentuados
  documento = ftfy.fix_text(documento, fix_entities=True)

  # Conta se existe \n no documento
  conta_caracter_barra_n = documento.count("\n")
  if conta_caracter_barra_n > 0:
    # Transforma \n em espaços em branco
    documento = documento.replace("\n"," ")
    conta_barra_n = conta_barra_n + 1

  # Conta se existe duas interrogações no documento
  conta_caracter_interrogacoes = documento.count("??")
  if conta_caracter_interrogacoes > 1:
    # Transforma 2 ou mais interrogações consecutivas em 1
    documento = re.sub(r"\?+", "?", documento)
    conta_interrogacoes = conta_interrogacoes + 1

  # Conta se existe duas exclamações no documento
  conta_caracter_exclamacoes = documento.count("!!")
  if conta_caracter_exclamacoes > 1:
    # Transforma 2 ou mais exclamações consecutivas em 1
    documento = re.sub(r"\!+", "!", documento)
    conta_exclamacoes = conta_exclamacoes + 1

  # Conta se existe duas vírgulas no documento
  conta_caracter_virgulas = documento.count(",,")
  if conta_caracter_virgulas > 1:
    # Transforma 2 ou mais vírgulas consecutivas em 1
    documento = re.sub(r"\,+", ",", documento)
    conta_virgulas = conta_virgulas + 1

  # Conta se existe tag no documento
  conta_tag = documento.count("<")
  if conta_tag > 0:
    # Remove as tags do documento
    documento = re.sub(r"<.*?>", "", documento)
    conta_tags = conta_tags + 1

  # Conta se existe dois caracteres branco
  conta_caracter_espacos = documento.count("  ")
  if conta_caracter_espacos > 0:
    # Transforma 2 ou mais espaços consecutivos em 1
    documento = re.sub(r"\ +", " ", documento)
    conta_espacos = conta_espacos + 1

  # >>> Outras operações de limpeza vão aqui! <<<

  # Guarda o id e o texto do documento tratado
  documentos_tratados.append([id_documento, documento])

INFO:root:Processando 40 documentos.


Dados:   0%|          | 0/40 [00:00<?, ?registro/s]

In [33]:
print("Total de documentos com \\n                        :", conta_barra_n)
print("Total de documentos com 2 ou mais interrogações   :", conta_interrogacoes)
print("Total de documentos com 2 ou mais exclamações     :", conta_exclamacoes)
print("Total de documentos com 2 ou mais vírgulas        :", conta_virgulas)
print("Total de documentos com tags                      :", conta_tags)
print("Total de documentos com 2 ou mais espaços         :", conta_espacos)
print("Total de documentos tratadas                      :", len(documentos_tratados))

Total de documentos com \n                        : 0
Total de documentos com 2 ou mais interrogações   : 0
Total de documentos com 2 ou mais exclamações     : 2
Total de documentos com 2 ou mais vírgulas        : 0
Total de documentos com tags                      : 0
Total de documentos com 2 ou mais espaços         : 0
Total de documentos tratadas                      : 40


Converte a lista em um pandas dataframe.

In [34]:
# Import das bibliotecas.
import pandas as pd

# Cria o dataframe da lista
df_documentos_tratados = pd.DataFrame(documentos_tratados, columns = ["id", "documento"])

#Mostra o número de documentos carregados
print("%d linhas carregadas do arquivo" % (len(df_documentos_tratados)))

# Mostra 10 linhas aleatórias dos dados
df_documentos_tratados.sample(10)

40 linhas carregadas do arquivo


,id,documento
13,14,Uma grande decepção para o que foi apresentado...
22,23,"Sou um grande fã de Emily Watson, Breaking The..."
36,37,"Eu entrei em uma livraria em Brentwood, Tennes..."
27,28,"Uhhh ... então, eles ainda têm escritores para..."
25,26,1980 foi certamente um ano para maus filmes de...
5,6,Uma coisa engraçada aconteceu comigo enquanto ...
15,16,Heres um decididamente médio post italiano apo...
0,1,"Mais uma vez, o Sr. Costner arrumou um filme p..."
10,11,Cage interpreta um bêbado e é elogiado pela cr...
38,39,Eu vi esse filme com minha namorada. Foi um de...


### 5.2.2 Descartando os documentos muito grandes

Função de descarte de documentos maiores que tamanho_maximo_token(512).

In [35]:
def descarteDocumentosGrandes(tamanho_maximo_token, dfdados):

  print('Tamanho do conjunto de dados antes do descarte: {}'.format(len(dfdados)))

  # Tokenize a codifica as setenças para o BERT.
  dfdados['input_ids'] = dfdados['documento'].apply(lambda tokens: tokenizer.encode(tokens, add_special_tokens=True))

  # Seleciona somente os documentos menores que tamanho_maximo_token
  dfdados = dfdados[dfdados['input_ids'].apply(len)<tamanho_maximo_token]

  print('Tamanho do conjunto de dados depois de do descarte: {}'.format(len(dfdados)))

  # Remove as colunas desnecessárias.
  dfdados = dfdados.drop(columns=['input_ids'])

  # Informações do DataFrame
  print(dfdados.info())

  return dfdados

Descartando documentos maiores que model_args.max_seq_len

In [36]:
df_documentos_tratados = descarteDocumentosGrandes(model_args.max_seq_len, df_documentos_tratados)

Tamanho do conjunto de dados antes do descarte: 40
Tamanho do conjunto de dados depois de do descarte: 36
<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 0 to 39
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         36 non-null     int64 
 1   documento  36 non-null     object
dtypes: int64(1), object(1)
memory usage: 864.0+ bytes
None


### 5.2.3 Segmentação do documento

Utiliza o spaCy para realizar a segmentação em sentenças do documento texto se model_args.sentenciar_documento for true.

In [37]:
# Import das bibliotecas.
import os # Biblioteca para acessar o sistema de arquivos
from tqdm.notebook import tqdm as tqdm_notebook # Biblioteca para barra de progresso

print("Processando",len(df_documentos_tratados),"documentos tratados")

# Sentenças por documento
lista_documentos_sentenciados = []
total_sentencas_documento = 0

logging.info("Processando {} documentos.".format(len(df_documentos_tratados)))

# Barra de progresso dos dados
dados_bar = tqdm_notebook(df_documentos_tratados.iterrows(), desc=f"Dados", unit=f"registro", total=len(df_documentos_tratados))

# Percorre os registros dos documento
for i, linha_documento in dados_bar:

  # Recupera o documento
  documento = str(linha_documento.values[1])

  # Aplica sentenciação do spacy no documento
  doc = nlp(documento)

  # Sequência das sentenças no documento
  sequencia = 1
  sentencas = []

  # Percorre as sentenças do documento
  for sentenca in doc.sents:

      # Conta as sentenças do documento
      total_sentencas_documento = total_sentencas_documento + 1

      # Transforma em string e remove os espaços do início e do fim
      sentenca1 = str(sentenca).strip()

      # Adiciona a sentença tratada na lista
      sentencas.append(str(sentenca1))

      # Incrementa a sequência da sentença no documento
      sequencia = sequencia + 1

  if model_args.sentenciar_documento == True:
    # Adiciona o documento na lista com a sentenciação
    lista_documentos_sentenciados.append([linha_documento.iloc[0], sentencas, str(linha_documento.iloc[1])])
  else:
    # Adiciona o documento na lista sem a sentenciação
    # A lista de sentenças é formado por um único texto, o próprio documento.
    lista_documentos_sentenciados.append([linha_documento.iloc[0], [str(linha_documento.iloc[1])], str(linha_documento.iloc[1])])


INFO:root:Processando 36 documentos.


Processando 36 documentos tratados


Dados:   0%|          | 0/36 [00:00<?, ?registro/s]

In [38]:
print("Total de documentos processados                                                :", len(lista_documentos_sentenciados))
print("Total sentenças nos documentos usando spaCy                                    :", total_sentencas_documento)

Total de documentos processados                                                : 36
Total sentenças nos documentos usando spaCy                                    : 437


## 5.3 Salva os dados processados

Gera o arquivo com os dados segmentados e limpos e depois compacta o arquivo para enviar para o Google Drive.

### 5.3.1 Especifica os nomes dos arquivos do dataset



In [39]:
# Nome do arquivo
NOME_ARQUIVO_DATASET = "dataset.csv"
NOME_ARQUIVO_DATASET_COMPACTADO = "dataset.zip"

### 5.3.2 Cria o arquivo do dataset

In [40]:
# Import das bibliotecas.
import pandas as pd

# Cria o dataframe da lista
df_lista_documentos_sentenciados = pd.DataFrame(lista_documentos_sentenciados, columns = ["id","sentencas","documento"])

df_lista_documentos_sentenciados.to_csv(DIRETORIO_LOCAL + NOME_ARQUIVO_DATASET, sep=";", index=False)

In [41]:
print(len(df_lista_documentos_sentenciados))

36


In [42]:
df_lista_documentos_sentenciados.sample(5)

,id,sentencas,documento
30,35,"[A presa tem uma história interessante, a meno...","A presa tem uma história interessante, a menos..."
24,27,[Tudo o que todo mundo já disse é verdade quan...,Tudo o que todo mundo já disse é verdade quand...
2,3,"[Primeiro de tudo eu odeio esses raps imbecis,...","Primeiro de tudo eu odeio esses raps imbecis, ..."
21,22,"[Isto é verdadeiramente, sem exagerar, um dos ...","Isto é verdadeiramente, sem exagerar, um dos p..."
19,20,[A família de Nova York é a última em seu bair...,A família de Nova York é a última em seu bairr...


### 5.3.3 Compacta e copia o dataset para uma pasta do GoogleDrive

Compacta os arquivos.

Usa o zip para compactar:
*   `-o` sobrescreve o arquivo se existir
*   `-j` Não cria nenhum diretório
*   `-q` Desliga as mensagens


In [43]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:

  !zip -o -q -j "$DIRETORIO_LOCAL$NOME_ARQUIVO_DATASET_COMPACTADO" "$DIRETORIO_LOCAL$NOME_ARQUIVO_DATASET"

Copia o arquivo compactado para o GoogleDrive



In [44]:
# Se estiver executando no Google Colaboratory
if IN_COLAB:
    # Copia o arquivo do dataset
    copiar_do_local_para_drive(NOME_ARQUIVO_DATASET_COMPACTADO)

    logging.info("Terminei a copia")


INFO:root:Arquivo copiado: /content/SRI/dataset.zip -> /content/drive/MyDrive/Colab Notebooks/SRI/data/dataset.zip
INFO:root:Terminei a copia


### 5.3.4 Carrega os dados

Realiza um teste carregando o arquivo do dataset criado.

In [45]:
# Import das bibliotecas.
import pandas as pd

# Abre o arquivo e retorna o DataFrame
df_dataset = pd.read_csv(DIRETORIO_LOCAL + NOME_ARQUIVO_DATASET, sep=";", encoding="UTF-8")

Corrigir o tipo de dados da lista dos documentos

Na lista
- coluna 1 - `sentenças` carregadas do arquivo vem como string e não como lista.

In [46]:
# Import das bibliotecas.
import ast # Biblioteca para conversão de string em lista

# Verifica se o tipo da coluna não é list e converte
df_dataset["sentencas"] = df_dataset["sentencas"].apply(lambda x: ast.literal_eval(x) if type(x)!=list else x)

logging.info("TERMINADO CORREÇÃO DOCUMENTOS: {}.".format(len(df_dataset)))

INFO:root:TERMINADO CORREÇÃO DOCUMENTOS: 36.


In [47]:
df_dataset.sample(5)

,id,sentencas,documento
17,18,[A terra foi destruída em um holocausto nuclea...,A terra foi destruída em um holocausto nuclear...
32,37,"[Eu entrei em uma livraria em Brentwood, Tenne...","Eu entrei em uma livraria em Brentwood, Tennes..."
4,5,[Filmes de fotos de latão não é uma palavra ap...,Filmes de fotos de latão não é uma palavra apr...
26,31,[Mesmo com os baixos padrões dos filmes de ter...,Mesmo com os baixos padrões dos filmes de terr...
29,34,"[Eu fui puxada para este filme no início, para...","Eu fui puxada para este filme no início, para ..."


# 6 Finalização

## 6.1 Tempo final de processamento



In [48]:
# Pega o tempo atual menos o tempo do início do processamento.
final_processamento = time.time()
tempo_total_processamento = formataTempo(final_processamento - inicio_processamento)

print("")
print("  Tempo processamento:  {:} (h:mm:ss)".format(tempo_total_processamento))


  Tempo processamento:  0:02:36 (h:mm:ss)
